In [ ]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame


---------------------------------------------------------------------------------


In [ ]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame

train_path = r'D:\LSTM-flare-prediction-1.0.2 - Copy\data\LSTM_C_sample_run\normalized_training.csv'
test_path = r'D:\LSTM-flare-prediction-1.0.2 - Copy\data\LSTM_C_sample_run\normalized_testing.csv'

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

print("Initial Train Shape:", train_data.shape)
print("Initial Test Shape:", test_data.shape)


def check_label(flare_label, flare):
    label = flare[0]  
    if flare_label == 'C':
        if label == 'X' or label == 'M':
            label = 'C'
        elif label == 'B':
            label = 'N'
    elif flare_label == 'M':
        if label == 'X':
            label = 'M'
        elif label == 'B' or label == 'C':
            label = 'N'
    elif flare_label == 'M5':
        if label == 'M':
            scale = flare[1:]
            if float(scale) >= 5.0:
                label = 'X'
            else:
                label = 'N'
        elif label == 'C' or label == 'B':
            label = 'N'  
    return label


def check_zero_record(flare_label, row):
    mapped_label = check_label(flare_label, row["flare"])

    if mapped_label == 'C':
        cols = [5, 7] + list(range(9, 13)) + list(range(14, 16)) + [18]
    elif mapped_label == 'M':
        cols = list(range(5, 10)) + list(range(13, 16)) + [19, 21] + list(range(23, 26))
    elif mapped_label == 'X':
        cols = list(range(5, 12)) + list(range(19, 21)) + list(range(22, 25))
    else:
        return True  
    for k in cols:
        if k >= len(row): continue
        if float(row.iloc[k]) == 0.0:
            return True 
    return False


def preprocess_data(df):
    df = df.copy()

    df["flare"] = df.apply(lambda row: check_label(row["flare"], row["flare"]), axis=1)
    print(df["flare"])

    df["timestamp"] = pd.to_datetime(df["timestamp"], format="%Y-%m-%dT%H:%M:%S.%fZ", errors="coerce")
    print (df["timestamp"])

    df = df.dropna(subset=["timestamp"])
    print(df.shape)

    df["item_id"] = df["NOAA"].astype(str)

    df["label"] = df["label"].map({"Positive": 1, "Negative": 0})

    label_mapping = {'C': 0, 'M': 1, 'X': 2, 'N': 3}
    df["flare_class"] = df["flare"].map(label_mapping)

    df["label"] = df["label"].astype(float)


    required_columns = [
        "item_id", "timestamp", "label", "flare", "NOAA", "HARP",
        "TOTUSJH", "Cdec", "TOTUSJZ", "Chis1d", "USFLUX", "TOTBSQ",
        "R_VALUE", "TOTPOT", "Chis", "SAVNCPP", "AREA_ACR",
        "Edec", "Xmax1d", "ABSNJZH"
    ]

    existing_columns = [col for col in required_columns if col in df.columns]
    
    missing_columns = set(required_columns) - set(existing_columns)
    if missing_columns:
        print(f" Warning: The following columns are missing and will be skipped: {missing_columns}")

    df = df[existing_columns]  

    return df



train_data_processed = preprocess_data(train_data)
test_data_processed = preprocess_data(test_data)

print("Processed Train Shape:", train_data_processed.shape)
print("Processed Test Shape:", test_data_processed.shape)

assert pd.api.types.is_datetime64_any_dtype(train_data_processed["timestamp"]), "Timestamp is NOT datetime64!"
assert pd.api.types.is_datetime64_any_dtype(test_data_processed["timestamp"]), "Timestamp is NOT datetime64!"

assert train_data_processed["item_id"].dtype == 'O', "item_id is NOT string!"
assert test_data_processed["item_id"].dtype == 'O', "item_id is NOT string!"

train_data_processed = TimeSeriesDataFrame(train_data_processed)
test_data_processed = TimeSeriesDataFrame(test_data_processed)

print("✅ Data successfully converted to TimeSeriesDataFrame!")


Initial Train Shape: (84577, 19)
Initial Test Shape: (185, 19)
0        N
1        N
2        N
3        N
4        N
        ..
84572    C
84573    C
84574    C
84575    C
84576    C
Name: flare, Length: 84577, dtype: object
0       2010-05-03 05:34:22.600
1       2010-05-03 06:34:22.600
2       2010-05-03 07:34:22.600
3       2010-05-03 08:34:22.600
4       2010-05-03 09:34:22.600
                  ...          
84572   2014-01-02 14:46:09.200
84573   2014-01-02 15:46:09.100
84574   2014-01-02 16:46:09.100
84575   2014-01-02 17:46:09.100
84576   2014-01-02 18:46:09.100
Name: timestamp, Length: 84577, dtype: datetime64[ns]
(84577, 19)
0      B
1      B
2      B
3      B
4      B
      ..
180    M
181    M
182    M
183    M
184    M
Name: flare, Length: 185, dtype: object
0     2017-03-25 01:22:36.700
1     2017-03-25 02:22:36.700
2     2017-03-25 04:22:36.800
3     2017-03-25 15:22:36.600
4     2017-03-25 16:22:36.600
                ...          
180   2017-04-02 07:22:37.900
181   2

In [13]:

print(train_data_processed.head)


<bound method NDFrame.head of                                  label flare   NOAA  HARP   TOTUSJH      Cdec  \
item_id timestamp                                                               
11063   2010-05-03 05:34:22.600    0.0     N  11063    11 -0.567834  0.000000   
        2010-05-03 06:34:22.600    0.0     N  11063    11 -0.571116  0.000000   
        2010-05-03 07:34:22.600    0.0     N  11063    11 -0.559336  0.000000   
        2010-05-03 08:34:22.600    0.0     N  11063    11 -0.584086  0.000000   
        2010-05-03 09:34:22.600    0.0     N  11063    11 -0.574319  0.000000   
...                                ...   ...    ...   ...       ...       ...   
11936   2014-01-02 14:46:09.200    1.0     C  11936  3535  2.873857  0.005635   
        2014-01-02 15:46:09.100    1.0     C  11936  3535  3.006697  0.005185   
        2014-01-02 16:46:09.100    1.0     C  11936  3535  2.975592  0.004770   
        2014-01-02 17:46:09.100    1.0     C  11936  3535  3.113931  0.004389  

In [14]:
print(train_data_processed.head)


<bound method NDFrame.head of                                  label flare   NOAA  HARP   TOTUSJH      Cdec  \
item_id timestamp                                                               
11063   2010-05-03 05:34:22.600    0.0     N  11063    11 -0.567834  0.000000   
        2010-05-03 06:34:22.600    0.0     N  11063    11 -0.571116  0.000000   
        2010-05-03 07:34:22.600    0.0     N  11063    11 -0.559336  0.000000   
        2010-05-03 08:34:22.600    0.0     N  11063    11 -0.584086  0.000000   
        2010-05-03 09:34:22.600    0.0     N  11063    11 -0.574319  0.000000   
...                                ...   ...    ...   ...       ...       ...   
11936   2014-01-02 14:46:09.200    1.0     C  11936  3535  2.873857  0.005635   
        2014-01-02 15:46:09.100    1.0     C  11936  3535  3.006697  0.005185   
        2014-01-02 16:46:09.100    1.0     C  11936  3535  2.975592  0.004770   
        2014-01-02 17:46:09.100    1.0     C  11936  3535  3.113931  0.004389  

In [ ]:
target = "label"
features = [col for col in train_data_processed.columns if col not in [target, "timestamp", "item_id"]]

# Define TimeLLM model
model = TimeLLM(h=12, input_size=len(features), max_steps=100)
nf = NeuralForecast(models=[model], freq='H')

# Train TimeLLM model
nf.fit(df=train_data_processed, val_size=12)

print("✅ TimeLLM model trained successfully!")


d:\flare-llm\autogluon_env\Lib\site-packages\autogluon\timeseries\predictor.py:197: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  std_freq = pd.tseries.frequencies.to_offset(self.freq).freqstr
Frequency 'H' stored as 'h'
Beginning AutoGluon training...
AutoGluon will save models to 'd:\flare-llm\lag-llama\chronos_flare_model'
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.12.3
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.22631
CPU Count:          16
GPU Count:          0
Memory Avail:       3.15 GB / 15.35 GB (20.5%)
Disk Space Avail:   9.14 GB / 48.83 GB (18.7%)
	We recommend a minimum available disk space of 10 GB, and large datasets may require more.

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'h',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 1,
 'q

In [ ]:
target = "label"

test_timestamps = test_data_processed["timestamp"].unique()
predicted_classes = []

test_data_copy = test_data_processed.copy()
test_data_copy = TimeSeriesDataFrame(test_data_copy)

def assign_label(mean_prob, flare_label):
        flare_label = str(flare_label)
        if (mean_prob >= 0.5 and flare_label.startswith("C")) or \
           (mean_prob >= 0.75 and flare_label.startswith("X")) or \
           (mean_prob >= 0.6 and flare_label.startswith("M")):
            return "Positive"
        return "Negative"



all_predictions = pd.DataFrame(columns=["timestamp", "item_id", "predicted_label"])
for t in range(len(test_timestamps)):  
    current_time = test_timestamps[t] 
    prediction = predictor.predict(test_data_copy.iloc[:t+1])
    prediction_mean = prediction["mean"].iloc[-1]  
    print(prediction_mean)
    fl = test_data_copy["flare"].iloc[t]
    print(fl)
    predicted_class = assign_label(prediction_mean, fl)
    predicted_classes.append(predicted_class)

# Print and verify predictions
test_data_processed["predicted_label"] = predicted_classes

output_path = "predicted_test_results.csv"
test_data_processed.to_csv(output_path, index=False)
print(f" Predictions saved to {output_path}")



data with frequency 'None' has been resampled to frequency 'h'.


Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble
data with frequency 'None' has been resampled to frequency 'h'.


-0.0005204104236304792
B


Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble
data with frequency 'None' has been resampled to frequency 'h'.


-0.0002572569937888565
B


...

1.0038413
M


data with frequency 'None' has been resampled to frequency 'h'.
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


1.0046276
M


data with frequency 'None' has been resampled to frequency 'h'.
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


1.005315
M
1.0060878
M
✅ Predictions saved to predicted_test_results.csv
